## Skeptic Training and Validation Set data preparation for Justification Finetuning

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

# pd.set_option("display.max_colwidth", None)

In [2]:
model_name = "amazon/MistralLite"

tokenizer = AutoTokenizer.from_pretrained(model_name)
print(f" PAD Token ID is set to - {tokenizer.pad_token_id}")
tokenizer.padding_side = "right"

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


 PAD Token ID is set to - 32000


### Loading Labelled data from GPT-4

In [3]:
df = pd.read_csv("data/claims_justification_205_records.csv")

In [4]:
df.head()

,video_id,title,author,description,org_transcript,eng_transcript,image_base64,url,Summary_Claims,Justification
0,FyQiu-o3VaI,IGL share price drop by 10.8%: Know the Reason...,5paisa,Indraprastha Gas share price (IGL) fell 10.84%...,"Hi Guys, IGL की Stock में आज 10% तक का बड़ा फॉ...","""Hi Guys, we have seen a big fall of up to 10%...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBggIDQgIDQ...,https://www.youtube.com/watch?v=FyQiu-o3VaI,The financial influencer claims that the IGL s...,The claim seems plausible as government polici...
1,TK74cx0p-NM,Nestle India share price jumps by over 1.5% af...,5paisa,Nestle India share price jumped more than 1.5%...,[Music] hi everyone 1.5 Financial results Mar...,[Music] Hi everyone! The financial results for...,/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBgcICAgIBw...,https://www.youtube.com/watch?v=TK74cx0p-NM,The financial influencer claims that the finan...,The claims made by the influencer are factual ...
2,70ifhSt5N_c,Wipro Q2 Results Highlights #q2results #shorts,5paisa,Discover the key highlights from Wipro's Q2 FY...,"Hi guys, Vipro ke numbers ek baar phir umeet s...","""Hi guys, Vipro's numbers have once again fall...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBggICAgICA...,https://www.youtube.com/watch?v=70ifhSt5N_c,The financial influencer claims that Vipro's r...,The claims made by the influencer are based on...
3,zb6dOEuW6qg,Nifty/BankNifty Prediction For Tomorrow for 19...,5paisa,Here you can find the nifty predictions for 19...,पिछले कुछ दिनों में मार्केट्स में ग्रैजूल रिका...,"In the past few days, the markets witnessed gr...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBgkICAgICQ...,https://www.youtube.com/watch?v=zb6dOEuW6qg,The financial influencer claims that the marke...,The claims made by the influencer are based on...
4,w-neJT7eaGU,Zomato Share Price Surges to 52-Week High: Kno...,5paisa,"On 18-Oct-2023, Zomato share price reached a 5...","Hi everyone, aaj Zomato ke shares ne a 52 week...","""Hi everyone, today Zomato's shares have hit a...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBgcICAgIBw...,https://www.youtube.com/watch?v=w-neJT7eaGU,The financial influencer claims that Zomato's ...,The claim seems plausible as partnerships with...


In [5]:
df.isna().sum()

video_id          0
title             0
author            0
description       5
org_transcript    0
eng_transcript    0
image_base64      8
url               0
Summary_Claims    0
Justification     0
dtype: int64

In [6]:
def prepare_prompt(sample):
    claim, justification = sample["Summary_Claims"], sample["Justification"]
    prompt = f"""You are a Financial Contrarian Writer. You are provided with a
Summary of Claims made by financial influencer and your task is to write
justifications for those claims. Claim Summary: {claim}"""
    text = ""
    text += f"<|prompter|>{prompt}</s><|assistant|>{justification}"
    return text

In [7]:
df["text"] = df.apply(prepare_prompt, axis=1)

In [8]:
train_data, test_ds = train_test_split(df, shuffle=True, test_size=0.1, random_state=42)
train_ds, val_ds = train_test_split(train_data, shuffle=True, test_size=0.1, random_state=42)

print(f"Train dataset {len(train_ds)}, and Validation dataset {len(val_ds)}, and Test dataset: {len(test_ds)}")

Train dataset 165, and Validation dataset 19, and Test dataset: 21


In [9]:
train_ds.head()

,video_id,title,author,description,org_transcript,eng_transcript,image_base64,url,Summary_Claims,Justification,text
124,dZ7xeVCYC5M,How I Would Invest $1000 If I Were In My 20s,The Game w/ Alex Hormozi,My new book $100M Leads is now LIVE. Grab your...,i [ __ ] guarantee you that you will be makin...,"I cannot guarantee you that, but I can assure ...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAUDBAgKDQoICg...,https://www.youtube.com/watch?v=dZ7xeVCYC5M\n,The influencer claims that investing in self-e...,The influencer's claim that investing in self-...,<|prompter|>You are a Financial Contrarian Wri...
1,TK74cx0p-NM,Nestle India share price jumps by over 1.5% af...,5paisa,Nestle India share price jumped more than 1.5%...,[Music] hi everyone 1.5 Financial results Mar...,[Music] Hi everyone! The financial results for...,/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBgcICAgIBw...,https://www.youtube.com/watch?v=TK74cx0p-NM,The financial influencer claims that the finan...,The claims made by the influencer are factual ...,<|prompter|>You are a Financial Contrarian Wri...
161,JnIYdowe9KE,Swing Trading Profit Double - Best Strategy,Stock Learners,In This Video I Have Share One Of My Trade Log...,तो स्टॉक मार्केट में कैसे आप स्विंग ट्रेडिंग क...,So how can you make a good return by swing tra...,/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAUDBBAQDxAQEB...,https://www.youtube.com/watch?v=JnIYdowe9KE\n,The financial influencer claims that by swing ...,The influencer's claim that one can make signi...,<|prompter|>You are a Financial Contrarian Wri...
198,GiiHU87xuGY,Top 3 positive stocks | Stocks for 23-Oct-2023...,PM Stock Academy,NaN,"वयलकम दोस्तो, बीयम स्टॉक अकडमी के एक नए वीडियो...","""Hello friends, welcome to the video of the Ve...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAUDBAgICAgICA...,https://www.youtube.com/watch?v=GiiHU87xuGY\n,The financial influencer suggests three stocks...,The claims made by the influencer are based on...,<|prompter|>You are a Financial Contrarian Wri...
178,gsXgM6WLJsg,"Turning $100 Into $1,000 Trading Stocks | Ep.1",Jenny Hoyos,get up to 10 FREE stocks (deposit at least $10...,this is a penny and last week i tried turning...,"""This is a penny, and last week I attempted to...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBg0KCQkJCQ...,https://www.youtube.com/watch?v=gsXgM6WLJsg\n,The financial influencer claims that he is att...,"The claim of turning 100 into 1,000 through st...",<|prompter|>You are a Financial Contrarian Wri...


In [10]:
train_ds.text[35]

"<|prompter|>You are a Financial Contrarian Writer. You are provided with a\nSummary of Claims made by financial influencer and your task is to write\njustifications for those claims. Claim Summary: The influencer claims that bond markets in India are gaining popularity and the Reserve Bank of India has introduced a new investment option - government securities with fifty years maturity. Previously, only ultra long-term duration bonds with a maturity of forty years were available. The influencer suggests that these bonds are considered safe and their demand is increasing. For retail investors, the influencer suggests that current deals in the secondary market could be attractive, despite limited liquidity for securities, and dead mutual funds could be an option.</s><|assistant|>The claim about the introduction of fifty years maturity government securities by the Reserve Bank of India is true as it is a matter of public record and can be verified from the RBI's official announcements. T

### Saving the train and validation sets

In [11]:
train_ds.to_csv("data/train_skeptic_df.csv", index=False)
val_ds.to_csv("data/val_skeptic_df.csv", index=False)

### Test Set Preparation

In [12]:
test_ds.drop(columns=["text"], inplace=True)

In [13]:
def test_prepare_prompt(sample):
    claim = sample["Summary_Claims"]
    prompt = f"""You are a Financial Contrarian Writer. You are provided with a
Summary of Claims made by financial influencer and your task is to write
justifications for those claims. Claim Summary: {claim}"""
    text = ""
    text += f"<|prompter|>{prompt}</s><|assistant|>"
    return text

In [14]:
test_ds["text"] = test_ds.apply(test_prepare_prompt, axis=1)

In [17]:
test_ds.to_csv("data/test_skeptic_df.csv", index=False)